# 6.5 

---
## 0. 환경 준비

In [ ]:
# Colab이면 드라이브 마운트 )

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print('PyTorch', torch.__version__, '| device:', 'cuda' if device == 0 else 'cpu')

In [ ]:
import os 

root_dir = '' # 경로복사

ANIMAL_PATH = os.path.join(root_dir, 'Dataset/animal.jpg')
CAR_PATH = os.path.join(root_dir, 'Dataset/car.png')
HORSE_PATH = os.path.join(root_dir, 'Dataset/horse.jpg')

In [ ]:
animal = Image.open(ANIMAL_PATH).convert('RGB')
car = Image.open(CAR_PATH).convert('RGB')
horse = Image.open(HORSE_PATH).convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(animal); axes[0].set_title('animal.jpg'); axes[0].axis('off')
axes[1].imshow(car); axes[1].set_title('car.png'); axes[1].axis('off')
axes[2].imshow(horse); axes[2].set_title('horse.jpg'); axes[2].axis('off')
plt.tight_layout(); plt.show()

---
## 1. 이미지 분류 (Image Classification)

ViT — ImageNet pretrained, **한 줄 pipeline**

In [ ]:
classifier = pipeline('image-classification', model='google/vit-base-patch16-224', device=device)
preds = classifier(animal)

print('Top-5 predictions:')
for p in preds[:5]:
    print(f"  {p['label']}: {p['score']:.3f}")

---
## 2. 객체 검출 (Object Detection)

DETR — bounding box + 클래스

In [ ]:
detector = pipeline('object-detection', model='facebook/detr-resnet-50', device=device)
detections = detector(car, threshold=0.5)

for d in detections:
    print(f"{d['label']}: {d['score']:.3f} | box={d['box']}")

In [ ]:
def draw_boxes(image, results):
    img = image.copy()
    draw = ImageDraw.Draw(img)
    for r in results:
        b = r['box']
        draw.rectangle([b['xmin'], b['ymin'], b['xmax'], b['ymax']], outline='red', width=3)
        draw.text((b['xmin'], max(0, b['ymin'] - 12)), f"{r['label']} {r['score']:.2f}", fill='red')
    return img

plt.figure(figsize=(8, 6))
plt.imshow(draw_boxes(car, detections))
plt.axis('off'); plt.title('Object Detection — car.png')
plt.show()

---
## 3. 인스턴스 분할 (Image Segmentation)

Mask2Former — 픽셀 단위 마스크

In [ ]:
segmenter = pipeline(
    'image-segmentation',
    model='facebook/mask2former-swin-tiny-coco-instance',
    device=device,
)
segments = segmenter(car)
print(f'{len(segments)} instances detected')
for s in segments[:5]:
    print(f"  {s['label']}: {s['score']:.3f}")

In [ ]:
def overlay_segments(image, results, alpha=0.45):
    base = np.array(image).astype(np.float32)
    overlay = base.copy()
    #cmap = plt.colormaps['tab20']
    cmap = plt.colormaps['autumn']
    for i, r in enumerate(results):
        mask = np.array(r['mask']) > 0
        color = np.array(cmap(i % 20)[:3]) * 255
        overlay[mask] = overlay[mask] * (1 - alpha) + color * alpha
    return overlay.astype(np.uint8)

In [ ]:
i=0

plt.figure(figsize=(8, 6))
plt.imshow(overlay_segments(car, segments[i:i+1]))
plt.axis('off'); plt.title(f'Segmentation — car - {segments[i]['label']}.jpg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(overlay_segments(car, segments))
plt.axis('off'); plt.title('Segmentation — car .jpg')
plt.show()

---
## 4. 이미지 캡션 (Image Captioning)

BLIP — 이미지를 문장으로 설명

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

In [ ]:
inputs = processor(horse, return_tensors="pt")

In [ ]:
outputs = model.generate(**inputs)
caption = processor.decode(outputs[0], skip_special_tokens=True)
print("Caption:", caption)